# DCT Laboratory — Volume I, Chapter 14
## Mathematical Analysis of the Unified Enterprise Transformation Architecture
**Seed `26114`** · Companion to Chapter 14 and **AXIOM Lab 1.14** (module **AXIOM-14**) · Mirrored in `DCT_V1_Ch14_Lab.xlsx`

This laboratory reproduces **Worked Example 14.1, *Can Meridian Afford the New Link?***, with the book's
declarations, and works the five steps of AXIOM Lab 1.14, *How Close to the Edge Is Meridian?*: the stability
bench, the sensitivity explorer, the perturbation simulator, the feasible-region visualizer and the diagnostics
dashboard, ending with the lab's deliverable, an analysis certificate for the board. The chapter's numbers are
computed exactly, without simulation; the lab's seed 26114 is used only for fresh draws, labelled as such. Every
Meridian number below is the book's or follows from its declarations (Worked Example 14.1, Table 14.4, Figures
14.1–14.3, Sections 14.4–14.6, Online Appendix 14.B.1–14.B.3); the final cell checks the printed ones. Money is
in \$ million.

## The declarations (Worked Example 14.1, the situation; Section 14.6)

**Meridian's feedback core** (Worked Example 9.1): Industrial Systems and Digital, each wired both ways to shared
engineering and to treasury. The risk committee's register (Section 12.7) caps every link at 0.40: no unit passes
on more than 40 percent of a quarter's deviation to a linked unit in the next quarter. Read as the interaction
matrix $\mathbf{M}$, the register's ceilings bound the true linearized dynamics, $|\mathbf{A}| \le \mathbf{M}$
entrywise (the declared-gain certificate, Equation 14.3). Where the book asks what happens *if every link ran at
its ceiling*, the deviations follow the register itself, $\mathbf{x}_{k+1} = \mathbf{M}\mathbf{x}_k$: Chapter 12's
quarterly echo.

**The new link.** The second phase of the digital program links Digital and Industrial Systems directly, both
ways: Digital's platform is sold into Industrial Systems' installed base, and Industrial Systems' service teams
carry Digital's software to the customers who are migrating. In the symmetric register the engineering links
carry a gain $e$, the treasury links $t$ and the new link $h$; today $e = t = 0.40$ and $h = 0$.

**The board's rule.** Any deviation must halve within a year: a half-life of at most four quarters, that is, a
spectral radius of at most $2^{-1/4} = 0.8409$.

**The runway** (Section 14.6; a unit, not Meridian). A unit holds \$120 million of cash above its floor, burns
\$50 million a quarter, and can cut the burn by at most \$10 million a quarter: $s' = s - r$ and
$r' = \max(r - u, 0)$ with $0 \le u \le 10$, on the feasible region $0 \le s \le 500$, $0 \le r \le 60$
(Appendix 14.A, proof of Theorem 14.5(iv)).

In [ ]:
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt
from math import log, sqrt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26114          # the lab's seed: fresh draws only, labelled as such

CORE = ["IND", "DIG", "ENG", "TRS"]       # Industrial Systems, Digital, engineering, treasury
G0 = 0.40                                  # the register's ceiling per link (Section 12.7)
RULE = 0.5 ** 0.25                         # the board's rule: a half-life of at most four quarters
SHOCK = 10.0                               # $10 million at Digital: a one-off shock, or a shortfall a quarter

def register(e=G0, t=G0, h=0.0):
    # the symmetric register on CORE: engineering links e, treasury links t, the new link h;
    # entry (i, j) is the share of unit j's deviation that unit i receives in the next quarter
    M = np.zeros((4, 4))
    i, d, g, r = range(4)
    for a, b, w in ((i, g, e), (d, g, e), (i, r, t), (d, r, t), (i, d, h)):
        M[a, b] = M[b, a] = w
    return M

def rho(M):
    return float(max(abs(np.linalg.eigvals(M))))

def rho_closed(e=G0, t=G0, h=0.0):
    # Online Appendix 14.B.1: rho = (h + sqrt(h^2 + 8(e^2 + t^2))) / 2
    return (h + sqrt(h * h + 8 * (e * e + t * t))) / 2

def half_life(r):
    return log(2) / log(1 / r) if r < 1 else float("inf")

def to_tenth(r):
    return log(10) / log(1 / r) if r < 1 else float("inf")

def amplification(r):
    return 1 / (1 - r) if r < 1 else float("inf")

def at(src="DIG", size=SHOCK):
    b = np.zeros(4); b[CORE.index(src)] = size
    return b

def settled(M, b=None):
    # Theorem 14.4(ii): a permanent input b a quarter settles at (I - M)^{-1} b; none when rho >= 1
    b = at() if b is None else b
    return np.linalg.solve(np.eye(4) - M, b) if rho(M) < 1 else None

def echo(M, b=None, quarters=12):
    # the deviation at each unit, quarter by quarter, after a one-off shock b (rows: quarters 0, 1, ...)
    x = at() if b is None else b
    out = [x]
    for _ in range(quarters):
        x = M @ x
        out.append(x)
    return np.array(out)

M0 = register()
print("today's register (rows receive, columns pass on):")
print("          " + "".join(f"{u:>6s}" for u in CORE))
for u, row in zip(CORE, M0):
    print(f"    {u:4s}  " + "".join(f"{v:6.2f}" for v in row))
print(f"the board's rule: half-life <= 4 quarters <=> rho <= 2^(-1/4) = {RULE:.4f}")

## Panel 1 — Read today's budget (Worked Example 14.1, Step 1; Theorems 14.3(ii), 14.3(iv) and 14.4(iii): 0.80, 3.1 quarters, \$50 million)

AXIOM Lab 1.14, step 1, the stability bench. Each core unit has two links, so it passes on at most 0.80 of a
deviation in total, and the register's spectral radius is **0.80**. By Theorem 14.3(ii) that certifies the core
whatever the true gains are, as long as each stays within its ceiling: a deviation halves every **3.1 quarters**.
Fresh draws of true gains within the ceilings, with the lab's seed, show the certificate at work. Its margin is the
gap to the edge, $1 - \rho = 0.20$, and 0.04 to the board's rule. The Lyapunov bowl of Theorem 14.3(iv), the
solution of $\mathbf{M}^{\top}\mathbf{P}\mathbf{M} - \mathbf{P} = -\mathbf{I}$, is positive definite, so the health
index $V(\mathbf{x}) = \mathbf{x}^{\top}\mathbf{P}\mathbf{x}$ falls every quarter. By Theorem 14.4(iii) the
amplification is $1/(1 - 0.80) = 5$: a permanent shortfall of \$10 million a quarter at Digital settles at
**\$50 million** a quarter across the core, \$19 million at Digital, \$11 million each at engineering and treasury,
and \$9 million at Industrial Systems, of which **\$40 million** is knock-on that Digital's own plan never shows.

In [ ]:
r0 = rho(M0)
print("each unit passes on (column sums):", ", ".join(f"{u} {v:.2f}" for u, v in zip(CORE, M0.sum(axis=0))))
print(f"eigenvalues {np.sort(np.linalg.eigvals(M0).real).round(4)}: spectral radius {r0:.4f}"
      f" (closed form sqrt(8 x 0.32)/2 = {rho_closed():.4f})")
print(f"half-life ln 2 / ln(1/0.80) = {half_life(r0):.2f} quarters; to a tenth {to_tenth(r0):.1f} quarters")
print(f"margin to the edge, 1 - rho = {1 - r0:.2f}; margin to the board's rule, {RULE:.4f} - {r0:.2f} = {RULE - r0:.4f}")

rng = np.random.default_rng(SEED)              # fresh draws, lab seed: true gains within the ceilings, |A| <= M
draws = np.array([rho(M0 * rng.uniform(-1.0, 1.0, size=M0.shape)) for _ in range(10_000)])
print(f"10,000 true Jacobians with |A| <= M (fresh draws, seed {SEED}): largest spectral radius {draws.max():.3f};"
      f" share at or below {r0:.2f}: {(draws <= r0 + 1e-12).mean():.0%}")

P0 = sla.solve_discrete_lyapunov(M0.T, np.eye(4))          # M'PM - P = -I
print(f"Lyapunov bowl: eigenvalues of P {np.linalg.eigvalsh(P0).round(3)}: positive definite;"
      f" steepest along the Perron direction, 1/(1 - rho^2) = {1 / (1 - r0 ** 2):.3f}")
xs = echo(M0)
V0 = np.einsum("ki,ij,kj->k", xs, P0, xs)
print("health index V(x_k) after a $10 million shock at Digital, quarters 0-6:", V0[:7].round(1),
      "falls every quarter:", bool((np.diff(V0) < 0).all()))

s0 = settled(M0)
print(f"amplification 1/(1 - {r0:.2f}) = {amplification(r0):.0f}")
print("a permanent $10 million a quarter shortfall at Digital settles at",
      ", ".join(f"{u} {v:.1f}" for u, v in zip(CORE, s0)),
      f"-> total {s0.sum():.1f}, of which {s0.sum() - SHOCK:.1f} knock-on")

## Panel 2 — Add the link at the ceiling (Worked Example 14.1, Step 2; Theorem 14.3(iii); Online Appendix 14.B.3: 1.0246, \$12 million, then \$15 million)

AXIOM Lab 1.14, step 1, continued: add the new link. Industrial Systems and Digital now have three links each and
pass on 1.20 of every deviation. The register's spectral radius rises to **1.02** (1.0246, Online Appendix
14.B.1), and it certifies nothing. If every link ran at its ceiling, a \$10 million shock at Digital would echo as
**\$12 million** in the next quarter and **\$15 million** after three years, instead of dying out (Figure 14.3).
Every link would still pass its own review at 0.40, and each unit alone is inert. Read in Chapter 9's two layers,
the divisions (Industrial Systems and Digital) and the corporate center (engineering and treasury), each layer is
stable on its own: the failure belongs to the two new loops, through engineering and through treasury. This is
Theorem 14.3(iii) at Meridian. The block sizes of part (iii), read layer by layer, certify today's core and not the
new one, and no positive definite Lyapunov bowl exists any more (part (iv)).

In [ ]:
M1 = register(h=G0)
r1 = rho(M1)
print("each unit passes on:", ", ".join(f"{u} {v:.2f}" for u, v in zip(CORE, M1.sum(axis=0))))
print(f"spectral radius {r1:.4f} (closed form (0.40 + sqrt(2.72))/2 = {rho_closed(h=G0):.4f}): above 1, no half-life")
adj0, adj1 = (M0 > 0).astype(int), (M1 > 0).astype(int)
print(f"loops of three links: today {np.trace(adj0 @ adj0 @ adj0) // 6}, with the new link {np.trace(adj1 @ adj1 @ adj1) // 6}"
      " (Industrial Systems-Digital-engineering and Industrial Systems-Digital-treasury)")

DIV, CTR = [0, 1], [2, 3]                  # Chapter 9's layers: the divisions and the corporate center
def block_sizes(M):
    # Theorem 14.3(iii): the size of each block, here the most one unit receives from the other layer (max-norm)
    size = lambda B: float(np.abs(B).sum(axis=1).max())
    return np.array([[size(M[np.ix_(a, b)]) for b in (DIV, CTR)] for a in (DIV, CTR)])
for name, M in (("today", M0), ("new link", M1)):
    Mh = block_sizes(M)
    print(f"{name:9s} divisions alone rho {rho(M[np.ix_(DIV, DIV)]):.2f}, center alone rho {rho(M[np.ix_(CTR, CTR)]):.2f};"
          f" block sizes {Mh.round(2).tolist()} -> rho {rho(Mh):.4f}")

e0, e1 = echo(M0).sum(axis=1), echo(M1).sum(axis=1)
print("echo of a $10 million shock at Digital, summed over the core, quarters 0-12")
print("  today   ", " ".join(f"{v:4.1f}" for v in e0))
print("  new link", " ".join(f"{v:4.1f}" for v in e1))
P1 = sla.solve_discrete_lyapunov(M1.T, np.eye(4))
print(f"Lyapunov equation with the new link: eigenvalues of P {np.linalg.eigvalsh(P1).round(2)}: indefinite, no bowl")

## Panel 3 — Re-budget under the board's rule (Worked Example 14.1, Step 3; Table 14.4; Online Appendix 14.B.1: 0.08, 0.16, 0.30)

AXIOM Lab 1.14, step 1, last part: find a set of trims that meets the board's rule. A half-life of at most four
quarters means a spectral radius of at most $r^{*} = 2^{-1/4} = 0.8409$. Because the core is symmetric, the
register's radius has a closed form in the engineering links $e$, the treasury links $t$ and the new link $h$,
$\rho = \tfrac{1}{2}\bigl(h + \sqrt{h^{2} + 8(e^{2} + t^{2})}\bigr)$, and $\rho \le r^{*}$ exactly when
$h\,r^{*} + 2(e^{2} + t^{2}) \le r^{*2}$ (Online Appendix 14.B.1): one frontier prices every repair. Running the new
link at no more than **0.08** keeps everything else but gives up most of the synergy. Ring-fencing treasury allows
the link at full strength if treasury's links fall to **0.16**. Trimming every existing link by a quarter, from 0.40
to **0.30**, does the same while spreading the cost across four offices. The radius sees the existing links only
through $e^{2} + t^{2}$, so the last two repairs have the same radius; they differ in where the settled deviation
lands.

In [ ]:
def h_max(e=G0, t=G0, r=RULE):
    return (r * r - 2 * (e * e + t * t)) / r                  # largest new link within the rule
def t_max(h=G0, e=G0, r=RULE):
    return sqrt((r * r - h * r) / 2 - e * e)                  # treasury's links that make room for h
def even_trim(h=G0, r=RULE):
    return sqrt((r * r - h * r) / 4)                          # one gain on all four existing links

OPTIONS = [("Today", G0, G0, 0.0), ("New link at the ceiling", G0, G0, G0),
           ("Run the new link light", G0, G0, h_max()), ("Ring-fence treasury", G0, t_max(), G0),
           ("Trim every existing link", even_trim(), even_trim(), G0)]
TABLE = []
print(f"Table 14.4                 {'eng.':>6s} {'treas.':>6s} {'new':>6s} {'rho':>7s} {'closed':>7s} {'half-life':>9s}"
      f" {'settled':>8s}")
for name, e, t, h in OPTIONS:
    M = register(e, t, h); r = rho(M); s = settled(M)
    TABLE.append(dict(name=name, e=e, t=t, h=h, M=M, rho=r, settled=s))
    hl = "none" if r >= 1 else f"{half_life(r):.1f}"
    st = "no limit" if s is None else f"{s.sum():.1f}"
    print(f"{name:26s} {e:6.3f} {t:6.3f} {h:6.3f} {r:7.4f} {rho_closed(e, t, h):7.4f} {hl:>9s} {st:>8s}")
print(f"the frontier h r* + 2(e^2 + t^2) <= r*^2, with r* = {RULE:.4f}:")
print(f"  e = t = 0.40:      h <= ({RULE ** 2:.4f} - 0.64)/{RULE:.4f} = {h_max():.3f}")
print(f"  h = e = 0.40:      t <= {t_max():.3f}")
print(f"  h = 0.40, e = t:   e <= {even_trim():.3f}, a trim of {1 - even_trim() / G0:.0%}")
print("  the new link allowed as treasury's links fall (engineering at 0.40):",
      ", ".join(f"t {t:.2f}: h {h_max(G0, t):.3f}" for t in (0.40, 0.30, 0.20, t_max(), 0.10, 0.0)))
ring, trim = TABLE[3]["settled"], TABLE[4]["settled"]
print("same radius, different landing: ring-fence", ", ".join(f"{u} {v:.1f}" for u, v in zip(CORE, ring)),
      "| trim", ", ".join(f"{u} {v:.1f}" for u, v in zip(CORE, trim)))
print("echo with every existing link trimmed, quarters 0-12:", " ".join(f"{v:.1f}" for v in echo(TABLE[4]["M"]).sum(axis=1)))

q = np.arange(13)
STYLE = {"Today": ("#1F4E79", "-", 2.0), "New link at the ceiling": ("#B71C1C", "-", 2.0),
         "Run the new link light": ("#616161", ":", 1.3), "Ring-fence treasury": ("#616161", "-.", 1.3),
         "Trim every existing link": ("#E65100", "--", 2.0)}
fig, ax = plt.subplots(figsize=(7.6, 3.6))
for row in TABLE:
    col, ls, lw = STYLE[row["name"]]
    ax.plot(q, echo(row["M"]).sum(axis=1), color=col, ls=ls, lw=lw, marker="o", ms=2.5,
            label=f"{row['name'].lower()}: ρ = {row['rho']:.2f}")
ax.set(xlabel="quarter after the shock", ylabel="deviation in the core (\\$ million)", xlim=(0, 12), ylim=(0, 16),
       title="A \\$10 million shock at Digital: the new link makes it grow (seed 26114)")
ax.legend(frameon=False, fontsize=8, loc="center right", bbox_to_anchor=(1.0, 0.5)); plt.tight_layout(); plt.show()

tt = np.linspace(0.0, G0, 201)
fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.plot(tt, [h_max(G0, t) for t in tt], color="#1F4E79", lw=2, label="trim treasury's two links (engineering at 0.40)")
ax.plot(tt, [h_max(t, t) for t in tt], color="#E65100", lw=2, ls="--", label="trim all four existing links alike")
ax.axhline(G0, color="#999999", lw=0.8)
ax.text(0.175, G0 + 0.012, "the new link at full strength, 0.40", fontsize=8, color="#616161")
for x, y, txt, col, dx, dy in ((G0, h_max(), f"run the link light: {h_max():.2f}", "#616161", -0.012, -0.065),
                               (t_max(), G0, f"ring-fence: {t_max():.2f}", "#1F4E79", -0.012, -0.075),
                               (even_trim(), G0, f"trim: {even_trim():.2f}", "#E65100", 0.008, 0.045)):
    ax.plot([x], [y], "o", color=col, ms=5, zorder=5)
    ax.text(x + dx, y + dy, txt, fontsize=8, color=col, ha="right" if dx < 0 else "left")
ax.set(xlabel="gain kept on the trimmed links (share passed on per link)", ylabel="largest new link within the rule",
       xlim=(0, 0.42), ylim=(0, 0.9), title="Every repair is a point on one frontier (seed 26114)")
ax.legend(frameon=False, fontsize=8, loc="upper right"); plt.tight_layout(); plt.show()

## Panel 4 — Price the nervousness (Worked Example 14.1, Step 4; Theorem 14.4(ii)–(iii); Figure 14.1: 5 to 6.3, \$64–70 million)

AXIOM Lab 1.14, step 2, the sensitivity explorer. The steady-state multiplier
$(\mathbf{I} - \mathbf{M})^{-1}\mathbf{B} = \mathbf{B} + \mathbf{M}\mathbf{B} + \mathbf{M}^{2}\mathbf{B} + \cdots$
splits the settled deviation into the direct effect that the parameter's owner sees and the knock-on effects along
walks of one, two, three links. Each repair, taken as far as the rule allows, sits on its edge: the amplification
rises from **5 to 6.3**, and the same permanent \$10 million shortfall at Digital now settles at
**\$64–70 million** a quarter, depending on the repair. What does not depend on the repair is the Perron-weighted
response, exactly the Perron-weighted direct effect divided by $1 - \rho$ (Theorem 14.4(iii)). The elasticity table
gives the unit-free sensitivities $(\theta_j/x_i)\,S_{ij}$ that offices should compare rather than raw derivatives.
Then turn the blow-up dial: scale today's gains toward a radius of 1 and record how fast the multipliers grow, like
$1/(1 - \rho)$, while the half-life stretches (Figure 14.1). The gap that slows recovery amplifies every
recalibration: near its edge a company is slow and nervous at once.

In [ ]:
B = at()
x, cum = B.copy(), np.zeros(4)
print("today's settled deviation, walk by walk (B, MB, M^2 B, ...): walks of n links")
print(f"  {'n':>3s} " + "".join(f"{u:>8s}" for u in CORE) + f"{'core':>8s}   cumulative core")
for n in range(9):
    cum += x
    print(f"  {n:3d} " + "".join(f"{v:8.3f}" for v in x) + f"{x.sum():8.3f}   {cum.sum():8.3f}")
    x = M0 @ x
print(f"  all  " + "".join(f"{v:8.3f}" for v in s0) + f"{s0.sum():8.3f}   direct {SHOCK:.0f}, knock-on {s0.sum() - SHOCK:.0f}")

print("\neach repair at the limit of the rule (Theorem 14.4(iii)):")
for row in TABLE[2:]:
    M, r, s = row["M"], row["rho"], row["settled"]
    vals, vecs = np.linalg.eig(M.T)
    w = np.abs(vecs[:, np.argmax(vals.real)].real); w = w / w[1]          # Perron weights, Digital = 1
    row["perron"] = float(w @ s)
    print(f"  {row['name']:26s} amplification {amplification(r):.2f}; settled total {s.sum():.1f};"
          f" Perron-weighted {w @ s:.2f} = {w @ B:.0f} / (1 - {r:.4f}); ||(I - M)^-1||_2 = {np.linalg.norm(np.linalg.inv(np.eye(4) - M), 2):.2f}")

def settled_closed(e, t, h, size=SHOCK):
    # by the IND-DIG symmetry: S = x_IND + x_DIG = size/(1 - h - 2(e^2 + t^2)), x_DIG - x_IND = size/(1 + h)
    S, d = size / (1 - h - 2 * (e * e + t * t)), size / (1 + h)
    return np.array([(S - d) / 2, (S + d) / 2, e * S, t * S])
def elasticities(e, t, h, eps=1e-6):
    x0, out = settled_closed(e, t, h), []
    for k, th in enumerate((e, t, h)):
        p, m = [e, t, h], [e, t, h]; p[k] += eps; m[k] -= eps
        dx = (settled_closed(*p) - settled_closed(*m)) / (2 * eps)
        out.append(np.append(th * dx / x0, th * dx.sum() / x0.sum()))
    return np.array(out).T
def rho_elasticities(e, t, h, eps=1e-6):
    out = []
    for k, th in enumerate((e, t, h)):
        p, m = [e, t, h], [e, t, h]; p[k] += eps; m[k] -= eps
        out.append(th * (rho_closed(*p) - rho_closed(*m)) / (2 * eps) / rho_closed(e, t, h))
    return np.array(out)
for row in (TABLE[0], TABLE[4]):
    el = elasticities(row["e"], row["t"], row["h"]); row["elasticities"] = el
    print(f"\nelasticities of the settled deviation, {row['name'].lower()} (percent per percent)")
    print(f"  {'':6s} {'engineering e':>14s} {'treasury t':>11s} {'new link h':>11s}")
    for name, line in zip(CORE + ["core"], el):
        print(f"  {name:6s} {line[0]:14.2f} {line[1]:11.2f} {line[2]:11.2f}")
    re = rho_elasticities(row["e"], row["t"], row["h"]); row["rho_elasticities"] = re
    print(f"  of the radius: {re[0]:.3f}, {re[1]:.3f}, {re[2]:.3f} (sum {re.sum():.2f}: the radius is homogeneous in the gains)")

print("\nthe blow-up dial: today's gains scaled by c")
print(f"  {'rho':>7s} {'c':>6s} {'amplification':>14s} {'half-life':>10s} {'settled total':>14s} {'at Digital':>11s}")
DIAL = []
for r in (0.80, RULE, 0.90, 0.95, 0.97, 0.99):
    c = r / r0; s = settled(c * M0)
    DIAL.append((r, amplification(r), half_life(r), s.sum()))
    print(f"  {r:7.4f} {c:6.3f} {amplification(r):14.2f} {half_life(r):10.1f} {s.sum():14.1f} {s[1]:11.1f}")

rr = np.linspace(0.30, 0.962, 400)
MARK = [(0.80, "today", "#1F4E79"), (RULE, "the board's limit", "#E65100"), (0.95, "near the edge", "#B71C1C")]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6))
ax.plot(rr, [half_life(v) for v in rr], color="#616161", lw=1.6)
bx.plot(rr, [amplification(v) for v in rr], color="#616161", lw=1.6)
short = lambda v: f"{v:.1f}".rstrip("0").rstrip(".")
for (v, lab, col), dy in zip(MARK, (-0.7, 0.7, 0.0)):
    ax.plot([v], [half_life(v)], "o", color=col, ms=5, zorder=5)
    bx.plot([v], [amplification(v)], "o", color=col, ms=5, zorder=5)
    if lab == "today":                      # below and right of its point, clear of the curve
        ax.text(v + 0.012, half_life(v) - 0.5, f"{lab}:\n{short(half_life(v))} quarters", fontsize=7.5, color=col,
                ha="left", va="top")
    else:
        ax.text(v - 0.03, half_life(v) + dy, f"{lab}: {short(half_life(v))} quarters", fontsize=7.5, color=col,
                ha="right", va="center")
    bx.text(v - 0.03, amplification(v) + dy, f"{lab}: {short(amplification(v))}×", fontsize=7.5, color=col, ha="right",
            va="center")
ax.set(xlabel="share left after a quarter, ρ", ylabel="half-life (quarters)", xlim=(0.3, 1.0), ylim=(0, 16),
       title="Recovery clock")
bx.set(xlabel="share left after a quarter, ρ", ylabel="amplification, 1/(1 − ρ)", xlim=(0.3, 1.0), ylim=(0, 24),
       title="Amplification")
for z in (ax, bx):
    z.tick_params(labelsize=7); z.xaxis.label.set_size(8); z.yaxis.label.set_size(8); z.title.set_size(9)
fig.suptitle("One gap, two clocks: slow and nervous at once (seed 26114)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 5 — Treat it as a structural change (Worked Example 14.1, Step 5; Definition 14.7; Theorem 14.4(i))

AXIOM Lab 1.14, step 3, the perturbation simulator. Definition 14.7 classifies a perturbation by what it changes,
and the three kinds propagate by different mathematics. A **state** perturbation, the \$10 million shock at
Digital, travels along the dynamics, and its echo is linear in its size. A **parametric** perturbation travels
along the sensitivity recursion $\mathbf{S}_{k+1} = \mathbf{A}_k\mathbf{S}_k + \mathbf{B}_k$ (Equation 14.4) toward
the steady-state multiplier, and a recalibrated ceiling is priced to first order by its derivative. A **structural**
perturbation changes the register itself: the new link is not a parameter of today's model, so no derivative of
today's model prices it, and the core must be re-analyzed with the new block. AXIOM preloads an acquisition as its
structural case; the book's own, with declared numbers, is the new link. The integration plan therefore becomes a
gain-reduction schedule: each milestone that switches on part of the synergy is matched by a recorded trim
elsewhere, re-certified against the rule, and checked against measured gains in the gain audit (Panel 7).

In [ ]:
e10, e20 = echo(M0).sum(axis=1), echo(M0, at(size=20.0)).sum(axis=1)
print("a state perturbation: the echo of $10 million and of $20 million at Digital, quarters 0-4:",
      e10[:5].round(2), e20[:5].round(2), f"-> ratio {(e20 / e10).min():.0f} in every quarter: linear in the shock")

S_k, path = np.zeros(4), {}
for k in range(1, 41):
    S_k = M0 @ S_k + B                                       # Equation 14.4 with A_k = M, B_k = $10 million at Digital
    path[k] = S_k.copy()
print("a parametric perturbation, the permanent shortfall: S_k by quarter (core total)",
      ", ".join(f"{k}: {path[k].sum():.2f}" for k in (1, 2, 4, 8, 16, 40)), f"-> (I - M)^-1 B = {s0.sum():.2f}")

e1_ = 0.39                                                   # recalibrate the engineering links' ceiling 0.40 -> 0.39
d = 1e-6
drho = (rho_closed(G0 + d) - rho_closed(G0 - d)) / (2 * d)
dtot = (settled_closed(G0 + d, G0, 0).sum() - settled_closed(G0 - d, G0, 0).sum()) / (2 * d)
RECAL = dict(drho=drho, dtot=dtot, rho_first=r0 + (e1_ - G0) * drho, rho_exact=rho_closed(e1_),
             tot_first=s0.sum() + (e1_ - G0) * dtot, tot_exact=settled_closed(e1_, G0, 0).sum())
print(f"a recalibration, the engineering links 0.40 -> 0.39: d rho/d e = {drho:.3f}, d total/d e = {dtot:.1f};"
      f" first order rho {RECAL['rho_first']:.4f}, total {RECAL['tot_first']:.2f};"
      f" re-analysis rho {RECAL['rho_exact']:.4f}, total {RECAL['tot_exact']:.2f}")

cum0 = np.cumsum(echo(M0, quarters=40).sum(axis=1)); cum1 = np.cumsum(echo(M1, quarters=40).sum(axis=1))
print("a structural perturbation, the new link: walk sums of the echo (cumulative deviation) after 4, 12 and 40 quarters:"
      f" today {cum0[4]:.1f}, {cum0[12]:.1f}, {cum0[40]:.1f} (limit {s0.sum():.0f});"
      f" with the new link {cum1[4]:.1f}, {cum1[12]:.1f}, {cum1[40]:.1f}, without limit")
dh_rho = (rho_closed(h=d) - rho_closed(h=-d)) / (2 * d)
dh_tot = (settled_closed(G0, G0, d).sum() - settled_closed(G0, G0, -d).sum()) / (2 * d)
TANGENT = dict(rho=r0 + G0 * dh_rho, total=s0.sum() + G0 * dh_tot)
print(f"even with the link written in as a parameter h, which is already the re-analysis, the tangent at h = 0 puts the"
      f" radius at {TANGENT['rho']:.2f} and the settled total at {TANGENT['total']:.1f}; the re-analysis gives"
      f" {r1:.4f} and no settled total: a structural change is not small")

## Panel 6 — Run the viability iteration (Theorem 14.5(iii)–(iv); Figure 14.2; Online Appendix 14.B.2: \$150 million to stop, out of cash in quarter 4)

AXIOM Lab 1.14, step 4, the feasible-region visualizer, on the corridor of Section 14.6. A unit holds \$120 million
of cash above its floor, burns \$50 million a quarter, and its commitments let it cut the burn by at most
\$10 million a quarter. Today is feasible, and so is next quarter whatever it does: cash will be \$70 million. Yet
stopping the burn takes $50 + 40 + 30 + 20 + 10 = 150$ of cash, and the unit has 120: with the fastest possible cuts,
cash runs out in the **fourth quarter**. The viability iteration of Theorem 14.5(iii) sees it in advance: start from
$V_0 = \mathcal{F}$ and keep in $V_{j+1}$ the states with a decision leading into $V_j$. The cell runs it on a grid of
states and decisions and finds each $V_j$ to be $\{s \ge N_j(r)\}$ with $N_j(r) = \sum_{i<j} \max(r - ib, 0)$
(Online Appendix 14.B.2): at a burn of 50 the requirement grows quarter by quarter through 50, 90, 120, 140 and 150,
the kernel's edge. Two repairs reopen the kernel: **\$30 million more cash**, or faster cuts, since exit clauses that
allow \$15 million a quarter make **\$110 million** enough (cuts of \$12.5 million would need \$125 million).

In [ ]:
CASH0, BURN0, CUT = 120.0, 50.0, 10.0
S_MAX, R_MAX = 500.0, 60.0

def runway_path(s, r, b=CUT, quarters=5, cut=True):
    out = [(0, s, r)]
    for k in range(1, quarters + 1):
        s, r = s - r, (max(r - b, 0.0) if cut else r)
        out.append((k, s, r))
    return out

def need(r, b=CUT, j=None):
    # N_j(r): the cash that j quarters of burn consume at the fastest cuts; N(r) as j grows
    tot, i = 0.0, 0
    while (j is None or i < j) and r - i * b > 1e-12:
        tot += r - i * b; i += 1
    return tot

def viability(b, ds=0.5):
    # Theorem 14.5(iii) on a grid: V_0 = F; V_{j+1} = states with some decision u in [0, b] leading into V_j
    S = np.arange(0.0, S_MAX + ds / 2, ds); R = np.arange(0.0, R_MAX + ds / 2, ds)
    i_s, i_r = np.meshgrid(np.arange(len(S)), np.arange(len(R)), indexing="ij")
    to_s = i_s - i_r                                   # s' = s - r, on the same grid
    V = [np.ones(i_s.shape, bool)]
    while True:
        nxt = np.zeros(i_s.shape, bool)
        for u in np.arange(0.0, b + ds / 2, ds):
            to_r = np.maximum(i_r - int(round(u / ds)), 0)
            nxt |= (to_s >= 0) & V[-1][np.clip(to_s, 0, None), to_r]
        if (nxt == V[-1]).all():
            return S, R, V
        V.append(nxt)

edge = lambda S, V, k: float(S[np.argmax(V[:, k])])     # the least cash in V at burn index k
print("the fastest cuts from 120 cash, 50 burn: quarter, cash, burn:",
      " | ".join(f"{k}: {s:.0f}, {r:.0f}" for k, s, r in runway_path(CASH0, BURN0, quarters=4)))
print(f"every decision this quarter leaves cash at {CASH0 - BURN0:.0f}: feasible now and next quarter")
S, R, V10 = viability(CUT)
ir = int(round(BURN0 / 0.5))
print(f"viability iteration, cuts of at most {CUT:.0f} a quarter: {len(V10) - 1} shrinking steps, then V_j stops changing")
print("  cash needed at a burn of 50 to stay admissible j quarters:",
      ", ".join(f"V_{j}: {edge(S, V, ir):.0f}" for j, V in enumerate(V10)))
gap = max(abs(edge(S, V, k) - need(r, CUT, j)) for j, V in enumerate(V10) for k, r in enumerate(R))
nested = all((V10[j + 1] <= V10[j]).all() for j in range(len(V10) - 1))
print(f"  the grid's edges equal N_j(r) at every burn, largest gap {gap:.1e}; nested V_(j+1) within V_j: {nested}")
member = [j for j, V in enumerate(V10) if V[int(round(CASH0 / 0.5)), ir]]
print(f"  (120, 50) lies in V_0 to V_{max(member)}, not in V_{max(member) + 1}: admissible for {max(member)} quarters, not forever")
print(f"  the kernel's edge N(r) = r(r + b)/(2b) where r is a multiple of b: N(50) = {need(BURN0):.0f}; N(30) = {need(30.0):.0f}")
KERNEL = {}
for b in (15.0, 12.5):
    Sb, Rb, Vb = viability(b)
    KERNEL[b] = edge(Sb, Vb[-1], ir)
    print(f"  cuts of at most {b:g} a quarter: the kernel's edge at a burn of 50 is {KERNEL[b]:.0f}"
          f" ({'enough' if KERNEL[b] <= CASH0 else 'not enough'} for 120)")
print(f"repair by cash: {need(BURN0) - CASH0:.0f} more, to {need(BURN0):.0f}")
print("no cuts at all: cash", ", ".join(f"{s:.0f}" for _, s, _ in runway_path(CASH0, BURN0, quarters=3, cut=False)),
      "-> it leaves the feasible region in the third quarter, from 20")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
grey = ["#C9CED6", "#AEB6C2", "#939DAD", "#78859A", "#5D6C86"]
for j in range(1, len(V10) - 1):
    ax.plot(R, [edge(S, V10[j], k) for k in range(len(R))], color=grey[min(j - 1, 4)], lw=1.0)
    ax.text(R[-1] + 0.4, edge(S, V10[j], len(R) - 1), f"$V_{j}$", fontsize=7, color="#616161", va="center")
ax.plot(R, [edge(S, V10[-1], k) for k in range(len(R))], color="#1F4E79", lw=2.2, label="the kernel's edge: cuts of at most 10")
Sb, Rb, Vb = viability(15.0)
ax.plot(Rb, [edge(Sb, Vb[-1], k) for k in range(len(Rb))], color="#E65100", lw=1.6, ls="--", label="the kernel's edge: cuts of at most 15")
ax.plot(R, R, color="#B71C1C", lw=1.0, label="below: out of cash next quarter")
pts = runway_path(CASH0, BURN0, quarters=4)
ax.plot([r for _, _, r in pts], [s for _, s, _ in pts], color="#000000", lw=1.0, marker="o", ms=3.5, zorder=5,
        label="fastest cuts from 120 cash, 50 burn")
for k, s, r in pts:
    ax.text(r + 1.0, s - 4, f"q{k}", fontsize=7, va="top", bbox=dict(fc="white", ec="none", pad=0.4), zorder=6)
ax.annotate("", xy=(BURN0, need(BURN0) - 2), xytext=(BURN0, CASH0 + 2), arrowprops=dict(arrowstyle="-|>", color="#1F4E79", lw=1.0))
ax.text(BURN0 - 1.0, 160, "+30 cash", fontsize=8, color="#1F4E79", ha="right")
ax.axhline(0, color="#999999", lw=0.6)
ax.set(xlabel="burn rate (\\$ million a quarter)", ylabel="cash above the floor (\\$ million)", xlim=(0, 64), ylim=(-40, 200),
       title="Only the kernel sees the runway trap in advance (seed 26114)")
ax.legend(frameon=False, fontsize=7.5, loc="upper left"); plt.tight_layout(); plt.show()

## Panel 7 — Read the dashboard and run the gain audit (Table 14.2; Definition 14.11; Theorem 14.6(ii))

AXIOM Lab 1.14, step 5, first part, the diagnostics dashboard. Table 14.2 lists five stability tests; the dashboard
runs them on the five registers of Table 14.4, read at their ceilings. The certified budget, the block small gain on
Chapter 9's two layers and the Lyapunov bowl agree with the spectral test, except that the block sizes are coarse for
the ring-fenced register: they certify its stability but not the board's rule. Contraction, the strongest and rarest
test, fails for the trimmed register although its radius meets the rule. The complexity panel of Definition 14.11
reads the register's dimension, sparsity, gap and conditioning. The gain audit of Theorem 14.6(ii) lays measured
gains against the register entry by entry and attributes each violation to the channel that exceeds its ceiling.
The book declares no measured record, so the measured gains here are those of three states Worked Example 14.1
describes. Against today's register the phase-two link is an undeclared channel, with a ceiling of zero, and the
audit flags it; against the re-budgeted register the audit passes only if the trims have actually happened.

In [ ]:
print("Table 14.2's tests on each register, at its ceilings (rho: spectral radius; L: max row sum)")
print(f"  {'register':26s} {'spectral':>9s} {'blocks':>7s} {'P: least eigenvalue':>20s} {'L':>6s} {'rule':>5s}")
TESTS = []
for row in TABLE:
    M = row["M"]; r = row["rho"]
    rb = rho(block_sizes(M))
    lam = np.linalg.eigvalsh(sla.solve_discrete_lyapunov(M.T, np.eye(4))).min()
    L = float(np.abs(M).sum(axis=1).max())
    TESTS.append(dict(name=row["name"], rho=r, blocks=rb, lyap_min=float(lam), L=L))
    print(f"  {row['name']:26s} {r:9.4f} {rb:7.4f} {lam:20.2f} {L:6.3f} {('yes' if r <= RULE + 1e-12 else 'no'):>5s}")
print("  each test passes below 1 (spectral, blocks, L) or above 0 (Lyapunov: P positive definite); the rule needs rho <= 0.8409")

print("\nthe complexity panel (Definition 14.11), one quarterly clock")
for row in TABLE:
    M = row["M"]; r = row["rho"]
    nz = int((M > 0).sum())
    gap = f"{1 - r:.3f}" if r < 1 else "none"
    cond = f"{amplification(r):.2f}" if r < 1 else "none"
    print(f"  {row['name']:26s} n = 4, channels {nz:2d} of 12, spectral gap {gap:>6s}, conditioning {cond:>5s}")

def audit(measured, ceiling, label):
    viol = [(CORE[i], CORE[j], measured[i, j], ceiling[i, j]) for i in range(4) for j in range(4)
            if abs(measured[i, j]) > ceiling[i, j] + 1e-9]
    print(f"{label}: {len(viol)} violation(s)" + ("; certificate renewed" if not viol else "; certificate suspended"))
    for i, j, m, c in viol:
        print(f"    channel {j} -> {i}: measured {m:.3f} against a ceiling of {c:.3f}")
    return viol

REG_TODAY, REG_NEW = register(), TABLE[4]["M"]          # today's register; the re-budgeted one (trim every link)
LIVE_TRIMMED, LIVE_UNTRIMMED = TABLE[4]["M"], register(h=G0)
print("\nthe gain audit (Theorem 14.6(ii)): measured gains against the register, entry by entry")
v1 = audit(LIVE_UNTRIMMED, REG_TODAY, "phase two switched on, measured against today's register")
reach = (np.linalg.matrix_power(np.eye(4) + adj1, 2) > 0)[:, [CORE.index("IND"), CORE.index("DIG")]].any(axis=1)
print(f"    its consequences travel along walks from that channel: every unit reached within two quarters: {bool(reach.all())}")
v2 = audit(LIVE_TRIMMED, REG_NEW, "the trims recorded and made, measured against the re-budgeted register")
v3 = audit(LIVE_UNTRIMMED, REG_NEW, "the link switched on but the trims never made, against the re-budgeted register")
print(f"    the guarantee fails with the audit: the register promises rho <= {rho(REG_NEW):.4f}; the measured gains give {rho(LIVE_UNTRIMMED):.4f}")

## Panel 8 — Trip an invariant on purpose (Theorem 14.7(iv); Section 14.7: a jump, a drift, a breach)

AXIOM Lab 1.14, step 5, last part. Along any admissible computed trajectory a declared identity holds exactly, and a
violation proves a fault (Theorem 14.7(iv)). The time signature of the violation guides the search: a jump suggests
an implementation error, a steady drift a mis-declared map, and a violation that arrives with a constraint breach an
inadmissible state. The runway's cash identity, cash falls each quarter by that quarter's burn, is tripped here on
purpose three ways along the fastest-cuts path: a quarter's burn not posted; cuts booked in the quarter they are
decided rather than the next; and a model that never lets cash fall below the floor. The classifier reads only the
residuals and the floor.

In [ ]:
def computed(fault):
    # the fastest-cuts path as a faulty model computes it; returns cash and burn by quarter
    s, r, out = CASH0, BURN0, [(CASH0, BURN0)]
    for k in range(5):
        r_next = max(r - CUT, 0.0)
        if fault == "unposted" and k == 1:
            s_next = s                                    # quarter 1's burn never posted
        elif fault == "early cuts":
            s_next = s - r_next                           # the cut booked in the quarter it is decided
        elif fault == "floored":
            s_next = max(s - r, 0.0)                      # cash never allowed below the floor
        else:
            s_next = s - r
        s, r = s_next, r_next
        out.append((s, r))
    return np.array(out)

def classify(path):
    s, r = path[:, 0], path[:, 1]
    resid = s[1:] - (s[:-1] - r[:-1])                     # the identity: s_(k+1) = s_k - r_k
    hit = np.nonzero(np.abs(resid) > 1e-9)[0]
    if len(hit) == 0:
        return resid, "identity holds"
    if (s[:-1][hit] - r[:-1][hit] < 0).all():
        return resid, "arrives with a breach of the floor: an inadmissible state"
    if len(hit) == 1:
        return resid, "a one-quarter jump: an implementation error"
    return resid, "a steady drift: a mis-declared map"

TRIP = {}
for fault in ("none", "unposted", "early cuts", "floored"):
    p = computed(fault)
    resid, verdict = classify(p)
    TRIP[fault] = dict(cash=p[:, 0].tolist(), resid=resid.tolist(), verdict=verdict)
    print(f"{fault:11s} cash {', '.join(f'{v:.0f}' for v in p[:, 0])}; residuals {', '.join(f'{v:+.0f}' for v in resid)}")
    print(f"{'':11s} -> {verdict}")
print("the unposted burn and the early cuts both hide the trap: the faulty books end above the floor")

## Panel 9 — Write the analysis certificate (AXIOM Lab 1.14, the deliverable)

The lab's deliverable is a one-page analysis certificate for the board: the register's spectral radius and its
margin; the half-life and the amplification; the re-budget for the new link, with its cost; the viability check and
its repairs; and the audit plan. The cell assembles it from the panels above. The audit plan measures first the
channels whose gains move the radius most, by the elasticities of Panel 4.

In [ ]:
lim = TABLE[4]
print("ANALYSIS CERTIFICATE, Meridian's feedback core (Worked Example 14.1; AXIOM Lab 1.14)")
print(f"1. Register today: spectral radius {r0:.2f}, margin {1 - r0:.2f} to the edge and {RULE - r0:.3f} to the board's rule;"
      f" certified for any true gains within the 0.40 ceilings.")
print(f"2. Clocks today: half-life {half_life(r0):.1f} quarters; amplification {amplification(r0):.0f}:"
      f" a permanent $10 million a quarter at Digital settles at ${s0.sum():.0f} million, ${s0.sum() - SHOCK:.0f} million knock-on.")
print(f"3. The new link at 0.40: radius {r1:.2f}, no certificate. Re-budget at the rule's limit, radius {RULE:.4f}:"
      f" the link at most {h_max():.2f}; or treasury's links to {t_max():.2f}; or every existing link to {even_trim():.2f}.")
print(f"   Cost: amplification {amplification(RULE):.1f} instead of 5; the same shortfall settles at"
      f" ${TABLE[2]['settled'].sum():.0f}-{TABLE[4]['settled'].sum():.0f} million a quarter.")
print(f"4. Viability: $120 million against a $50 million burn cut by at most $10 million a quarter is admissible for"
      f" {max(member)} quarters; stopping needs ${need(BURN0):.0f} million. Repairs: +${need(BURN0) - CASH0:.0f} million,"
      f" or cuts of $15 million a quarter (need ${KERNEL[15.0]:.0f} million).")
re = lim["rho_elasticities"]
print(f"5. Audit plan (trim every existing link): measure the new link first, elasticity of the radius {re[2]:.2f},"
      f" then the four trimmed links, {re[0] / 2:.2f} each, to know that the trims have actually happened; compare each"
      f" with its recorded ceiling every quarter; the risk office, which owns the register, suspends the certificate"
      f" when a channel exceeds its ceiling.")

## Validation — the book's numbers, and agreement with `DCT_V1_Ch14_Lab.xlsx`

Each checkpoint is a number printed in Chapter 14 (Worked Example 14.1, Table 14.4, Figures 14.1–14.3, Sections
14.4–14.6, LOS Guidance 14.3–14.4) or in Online Appendix 14.B.1–14.B.3. The workbook recomputes the ones marked
*workbook* with Excel formulas and shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
T = {row["name"]: row for row in TABLE}
light, ring_, trim_ = T["Run the new link light"], T["Ring-fence treasury"], T["Trim every existing link"]
e_trim = echo(trim_["M"]).sum(axis=1)
checks = [
    ("today: pass-on per unit (workbook)", M0.sum(axis=0).max(), 0.80, 1e-9),
    ("today: spectral radius (workbook)", r0, 0.80, 5e-5),
    ("today: half-life, quarters (workbook)", half_life(r0), 3.11, 0.005),
    ("today: quarters to a tenth (workbook)", to_tenth(r0), 10.3, 0.05),
    ("today: amplification (workbook)", amplification(r0), 5, 1e-9),
    ("today: settled total, $ million (workbook)", s0.sum(), 50, 0.05),
    ("today: settled at Digital (workbook)", s0[1], 18.89, 0.005),
    ("today: settled at engineering (workbook)", s0[2], 11.11, 0.005),
    ("today: settled at treasury (workbook)", s0[3], 11.11, 0.005),
    ("today: settled at Industrial Systems (workbook)", s0[0], 8.89, 0.005),
    ("today: knock-on, $ million (workbook)", s0.sum() - SHOCK, 40, 0.05),
    ("today: echo in quarter 3 (workbook)", e0[3], 5.1, 0.05),
    ("today: echo in quarter 12 (workbook)", e0[12], 0.7, 0.05),
    ("new link: pass-on of IND and DIG (workbook)", M1.sum(axis=0)[1], 1.20, 1e-9),
    ("new link: spectral radius (workbook)", r1, 1.0246, 5e-5),
    ("new link: new loops of three links", np.trace(adj1 @ adj1 @ adj1) / 6, 2, 0),
    ("new link: echo in quarter 1 (workbook)", e1[1], 12, 0.05),
    ("new link: echo in quarter 3 (workbook)", e1[3], 12.2, 0.05),
    ("new link: echo after three years (workbook)", e1[12], 14.8, 0.05),
    ("board's rule: spectral radius (workbook)", RULE, 0.8409, 5e-5),
    ("run the link light: new link (workbook)", light["h"], 0.080, 5e-4),
    ("ring-fence: treasury links (workbook)", ring_["t"], 0.159, 5e-4),
    ("trim: every existing link (workbook)", trim_["e"], 0.304, 5e-4),
    ("at the limit: half-life, quarters (workbook)", half_life(trim_["rho"]), 4.0, 0.05),
    ("at the limit: quarters to a tenth (workbook)", to_tenth(trim_["rho"]), 13.3, 0.05),
    ("at the limit: amplification (workbook)", amplification(trim_["rho"]), 6.29, 0.005),
    ("run the link light: settled total (workbook)", light["settled"].sum(), 64.2, 0.05),
    ("ring-fence: settled total (workbook)", ring_["settled"].sum(), 68.0, 0.05),
    ("trim: settled total (workbook)", trim_["settled"].sum(), 70.2, 0.05),
    ("trim: echo in quarter 1 (workbook)", e_trim[1], 10.1, 0.05),
    ("trim: echo in quarter 12 (workbook)", e_trim[12], 1.4, 0.05),
    ("near the edge, 0.95: half-life (workbook)", half_life(0.95), 13.5, 0.05),
    ("near the edge, 0.95: amplification (workbook)", amplification(0.95), 20, 1e-9),
    ("runway: cash after one quarter (workbook)", runway_path(CASH0, BURN0)[1][1], 70, 0),
    ("runway: cash in quarter 4, fastest cuts (workbook)", runway_path(CASH0, BURN0)[4][1], -20, 0),
    ("runway: quarter cash runs out, fastest cuts (workbook)", next(k for k, s, _ in runway_path(CASH0, BURN0) if s < 0), 4, 0),
    ("runway: N_3(50), cash for three quarters (workbook)", need(BURN0, j=3), 120, 0),
    ("runway: N_4(50), cash for four quarters (workbook)", need(BURN0, j=4), 140, 0),
    ("runway: cash needed to stop (workbook)", need(BURN0), 150, 0),
    ("runway: more cash needed (workbook)", need(BURN0) - CASH0, 30, 0),
    ("runway: kernel edge, grid iteration", edge(S, V10[-1], ir), 150, 0),
    ("runway: quarters admissible from (120, 50) (workbook)", max(member), 3, 0),
    ("runway: need with cuts of 15 (workbook)", need(BURN0, 15.0), 110, 0),
    ("runway: need with cuts of 12.5 (workbook)", need(BURN0, 12.5), 125, 0),
    ("runway: kernel edge with cuts of 15, grid", KERNEL[15.0], 110, 0),
    ("runway: no cuts, cash after two quarters (workbook)", runway_path(CASH0, BURN0, cut=False)[2][1], 20, 0),
    ("runway: no cuts, quarter it leaves the region (workbook)",
     next(k for k, s, _ in runway_path(CASH0, BURN0, cut=False) if s < 0), 3, 0),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 15 poses the General Enterprise Optimization Problem on this certified object: its feasible set is
the audited region with the viability kernel as its honest core, its gradients are this chapter's sensitivities, and
under the board's half-life rule the stability budget is a shared resource with a price: every 0.01 trimmed from both
engineering links, or from both treasury links, frees 0.019 of the new link. AXIOM-14 runs the same model
interactively (Lab 1.14); this notebook and the workbook are its reproducible record.